# Entropy-Constrained GPTQ Allocator

April 9 SP8192 3-layer recurrence stack with a compressed-size-aware GPTQ export allocator.

The SP8192 shards are hosted in `kevclark/parameter-golf`; `run.sh` selects that dataset repo and refreshes stale manifests automatically.

In [ ]:
!git clone https://github.com/IanniMuliterno/parameter-golf.git || true
%cd parameter-golf/colab/2026-04-14_EntropyConstrained_GPTQ_Allocator

## April 9 Winner-Compatible Run

These values mirror `records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT/train_seed42.log` as closely as this Colab wrapper can. It uses the legacy fixed GPTQ export (`MATRIX_BITS=6`, `EMBED_BITS=8`) instead of the April 14 entropy allocator.

This profile is intended for a large GPU setup. On a T4, use the smoke-test cell below.

In [ ]:
import os

winner_env = {
    "MATCHED_FINEWEB_REPO_ID": "kevclark/parameter-golf",
    "RECORD_PROFILE": "1",
    "TRAIN_SHARDS": "128",
    "NPROC_PER_NODE": "1",  # Use "8" on an 8xH100 box to match the winner log.
    "EXPORT_ALLOCATOR": "legacy",
    "COMPUTE_DTYPE": "bf16",
    "ENABLE_COMPILE": "1",
    "SEED": "42",
    "ITERATIONS": "20000",
    "WARMDOWN_FRAC": "0.72",
    "WARMUP_STEPS": "20",
    "TRAIN_BATCH_TOKENS": "786432",
    "TRAIN_SEQ_LEN": "2048",
    "TRAIN_LOG_EVERY": "500",
    "MAX_WALLCLOCK_SECONDS": "600",
    "VAL_BATCH_TOKENS": "524288",
    "EVAL_SEQ_LEN": "2048",
    "VAL_LOSS_EVERY": "4000",
    "SLIDING_WINDOW_ENABLED": "1",
    "VOCAB_SIZE": "8192",
    "NUM_LAYERS": "11",
    "XSA_LAST_N": "11",
    "MODEL_DIM": "512",
    "EMBEDDING_DIM": "512",
    "NUM_KV_HEADS": "4",
    "NUM_HEADS": "8",
    "MLP_MULT": "4.0",
    "SKIP_GATES_ENABLED": "1",
    "TIE_EMBEDDINGS": "1",
    "LOGIT_SOFTCAP": "30.0",
    "ROPE_BASE": "10000.0",
    "ROPE_DIMS": "16",
    "ROPE_TRAIN_SEQ_LEN": "2048",
    "LN_SCALE": "1",
    "QK_GAIN_INIT": "5.25",
    "NUM_LOOPS": "2",
    "LOOP_START": "3",
    "LOOP_END": "5",
    "ENABLE_LOOPING_AT": "0.35",
    "PARALLEL_RESIDUAL_START": "7",
    "MIN_LR": "0.0",
    "EMBED_LR": "0.6",
    "HEAD_LR": "0.008",
    "TIED_EMBED_LR": "0.03",
    "TIED_EMBED_INIT_STD": "0.005",
    "MATRIX_LR": "0.022",
    "SCALAR_LR": "0.02",
    "MUON_MOMENTUM": "0.99",
    "MUON_BACKEND_STEPS": "5",
    "MUON_MOMENTUM_WARMUP_START": "0.92",
    "MUON_MOMENTUM_WARMUP_STEPS": "1500",
    "MUON_ROW_NORMALIZE": "1",
    "BETA1": "0.9",
    "BETA2": "0.95",
    "ADAM_EPS": "1e-08",
    "GRAD_CLIP_NORM": "0.3",
    "EVAL_STRIDE": "64",
    "MUON_BETA2": "0.95",
    "ADAM_WD": "0.02",
    "MUON_WD": "0.095",
    "EMBED_WD": "0.085",
    "EMA_DECAY": "0.9965",
    "TTT_ENABLED": "1",
    "TTT_LR": "0.005",
    "TTT_EPOCHS": "3",
    "TTT_MOMENTUM": "0.9",
    "TTT_CHUNK_TOKENS": "32768",
    "ETLB_ENABLED": "0",
    "ETLB_LR": "0.05",
    "ETLB_STEPS": "5",
    "ETLB_CLIP": "3.0",
    "COMPRESSOR": "brotli",
    "GPTQ_CALIBRATION_BATCHES": "64",
    "GPTQ_RESERVE_SECONDS": "12.0",
    "MATRIX_BITS": "6",
    "EMBED_BITS": "8",
    "MATRIX_CLIP_SIGMAS": "12.85",
    "EMBED_CLIP_SIGMAS": "20.0",
    "CONTROL_TENSOR_NAME_PATTERNS": "attn_scale,attn_scales,mlp_scale,mlp_scales,resid_mix,resid_mixes,q_gain,skip_weight,skip_weights,skip_gates",
}

os.environ.update(winner_env)
!INSTALL_DEPS=1 DOWNLOAD_DATA=1 bash run.sh

In [ ]:
# Faster allocator smoke-test variant for a Colab T4.
# This intentionally changes the record settings to fit cheaper Colab hardware.
# !RECORD_PROFILE=0 TRAIN_SHARDS=10 NPROC_PER_NODE=1 COMPUTE_DTYPE=fp16 ENABLE_COMPILE=0 PG_COLAB_DISABLE_COMPILE=1 TTT_ENABLED=0 EXPORT_ALLOCATOR=entropy ALLOCATOR_MATRIX_BITS=5,6 ALLOCATOR_MATRIX_SIGMAS=12.85 GPTQ_CALIBRATION_BATCHES=4 bash run.sh